# Peaked Circuit Solver

Solve peaked quantum circuits using MPS (Matrix Product State) or general tensor network contraction.

**Only dependency: numpy** (pre-installed on Colab).

Colab provides **12 GB RAM** (free) or **50+ GB** (Pro), which handles circuits that run out of memory locally.

In [ ]:
# Clone the repo (run once)
!git clone https://github.com/hosseinsadeghi/peaked-solver.git 2>/dev/null || echo 'Already cloned'
%cd peaked-solver

In [ ]:
import time
from pathlib import Path
from peaked_solver import QuantumCircuit, mps_solve, tn_solve

# List available circuits
circuits = sorted(Path('circuits').glob('*.qasm'))
for f in circuits:
    qasm = f.read_text()
    c = QuantumCircuit.from_openqasm(qasm)
    print(f'{f.name:<35} {c.n_qubits:>4}q  {len(c.gates):>6} gates  depth {c.depth()}')

## Solve a single circuit

In [ ]:
# Pick a circuit
QASM_FILE = 'circuits/P3_sharp_peak.qasm'  # <-- change this
SOLVER    = 'mps'      # 'mps' or '2d'
TOP_K     = 10
CHI_MAX   = None       # None = auto, or set e.g. 128, 256
HEAVY_HEX = False      # True for IBM heavy-hex topology circuits

qasm_str = Path(QASM_FILE).read_text()
circuit = QuantumCircuit.from_openqasm(qasm_str)
print(f'Circuit: {Path(QASM_FILE).name}')
print(f'Qubits:  {circuit.n_qubits}')
print(f'Gates:   {len(circuit.gates)}')
print(f'Depth:   {circuit.depth()}')
print()

In [ ]:
t0 = time.time()

if SOLVER == 'mps':
    kwargs = {'top_k': TOP_K, 'heavy_hex': HEAVY_HEX}
    if CHI_MAX is not None:
        kwargs['chi_max'] = CHI_MAX
    result = mps_solve(circuit, **kwargs)
else:
    result = tn_solve(circuit, top_k=TOP_K, timeout=120.0)

wall = time.time() - t0

print(f'Solver:     {result.heuristic_used}')
print(f'Time:       {result.compute_time_ms:.0f} ms (wall: {wall:.1f}s)')
print(f'Accuracy:   {result.accuracy_estimate:.4e}')
print(f'Trunc err:  {result.truncation_error:.2e}')
print(f'Analysis:   {result.circuit_analysis}')
print()

# Display top-k
n = circuit.n_qubits
bitstrings = result.top_k_bitstrings
max_prob = bitstrings[0][1] if bitstrings else 0

print(f'{"Rank":<6} {"Bitstring":<{n+4}} {"Probability":>12}  Bar')
print('-' * (30 + n))
for i, (bs, prob) in enumerate(bitstrings, 1):
    bar = chr(0x2588) * int(prob / max_prob * 40) if max_prob > 0 else ''
    display = bs if len(bs) <= 56 else bs[:28] + '...' + bs[-28:]
    print(f'{i:<6} {display:<{n+4}} {prob:>12.6f}  {bar}')

total = sum(p for _, p in bitstrings)
print(f'\nTop-{len(bitstrings)} total probability: {total:.6f}')

## Batch solve all circuits

In [ ]:
import traceback

results = {}
for qasm_file in sorted(Path('circuits').glob('*.qasm')):
    name = qasm_file.stem
    qasm_str = qasm_file.read_text()
    circuit = QuantumCircuit.from_openqasm(qasm_str)
    
    # Auto-select: heavy-hex flag for heavy_hex circuits
    heavy_hex = 'heavy_hex' in name.lower()
    
    print(f'\n{"=" * 60}')
    print(f'{name}: {circuit.n_qubits}q, {len(circuit.gates)} gates, depth {circuit.depth()}')
    
    try:
        t0 = time.time()
        result = mps_solve(circuit, top_k=5, heavy_hex=heavy_hex)
        wall = time.time() - t0
        
        results[name] = result
        print(f'  Time: {result.compute_time_ms:.0f}ms (wall: {wall:.1f}s)')
        print(f'  Accuracy: {result.accuracy_estimate:.4e}, Trunc: {result.truncation_error:.2e}')
        for i, (bs, prob) in enumerate(result.top_k_bitstrings[:3], 1):
            display = bs if len(bs) <= 40 else bs[:20] + '...' + bs[-20:]
            print(f'  #{i}: {display}  p={prob:.6f}')
    except Exception as e:
        print(f'  FAILED: {e}')
        traceback.print_exc()

## CLI usage (from terminal)

You can also run from the command line:

In [ ]:
!python solve.py circuits/P1_little_peak.qasm --solver mps --top-k 5

In [ ]:
!python solve.py circuits/P7_heavy_hex_1275.qasm --solver mps --heavy-hex --chi-max 128